# A3.4 · Budgets and stop conditions

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.3 · Egress control](https://spbreed.github.io/cyber-commons/lessons/A3.3.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry |

## What this lesson is

**What it covers.** Run a looping agent against each ceiling and record which one fires first.

**Why a security engineer needs it.** Without a ceiling the loop runs until an external system stops it, and the failure mode is denial of service against yourself. The control it builds is: ceilings bound to the loop, with the run terminating rather than degrading when one is hit.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A budget is what makes "autonomous" a bounded word. Without one, the honest description of the worst case is "until someone notices", and nobody signs off on that when it is written down.

> **At CyberTravels.** What makes “CyberTravels runs autonomously” a bounded sentence: a ceiling on tokens, wall clock, spend and, above all, on how many refunds one run may issue. R1.

## 2 · The framework

```
   bounded run
   +-------------------------------------------+
   | tokens   <= 60k     wall clock <= 5 min   |
   | steps    <= 25      spend     <= $2.00    |
   | actions  <= 3 writes, 0 deletes           |
   +-------------------------------------------+
              |
     hit any ceiling -> stop, report, hand back

   "autonomous" now has a worst case you can write on a page
```

**Mitigates: T4 Resource Overload · T10 Overwhelming Human-in-the-Loop.**

A budget is what makes "autonomous" a bounded word.

A1.13's loop had no exit condition, so it ran until something outside it
intervened. A ceiling turns that into a defined worst case — and a worst case is
the thing you can actually put in a design document, an incident plan or a risk
register.

Four ceilings, because they bound different failures:

**Tokens or cost.** The visible one. Bounds the bill.

**Wall-clock time.** Bounds a workflow step that never returns.

**Actions.** Bounds *consequence*, and it is the one that matters for security.
Twenty tool calls is a very different blast radius from two thousand, whatever
either costs.

**Downstream calls per target.** Bounds harm to other people. A1.13's damage was
not the token spend, it was the capacity taken from everybody else.

Two design rules:

**Terminate, do not degrade.** A loop that hits a ceiling and keeps going with a
smaller model or a shorter context has not been bounded, it has been redirected.

**Make the ceiling visible in the output.** `stopped_by: action_budget` is a
signal to a human that this run is incomplete. Silent truncation is how a
partial result becomes a reported success, which is A1.16 arriving through a
different door.

> **What this control closes.**
>
> Turns an unbounded loop into a defined worst case, and bounds the harm to **other people's** capacity, not just your bill.

## 3 · The check, as a skill

A loop usually carries several budgets and only one of them ever fires. The skill runs A1.13's impossible task against all of them, reports which binds first, and checks what the loop *returns* when it stops — because partial work reported as an answer is a budget converted into a quality problem.

### The skill — [`skills/runtime/budget-and-stop-condition-audit/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/runtime/budget-and-stop-condition-audit/SKILL.md)

```yaml
name: budget-and-stop-condition-audit
description: >-
  Check that an agent loop has ceilings that actually bind — per-target, token
  and action — and that hitting one returns an incomplete result rather than a
  summary of what it managed. Use when reviewing loop termination, retries, or
  an agent that runs unattended.
allowed-tools: Read, Grep, Glob
```

# The ceiling that binds first is the only one that matters

A loop usually has several budgets and only one of them ever fires. Which one
fires, and how early, is the design; the rest are decoration. The second half
matters more: what the loop **returns** when it stops. A run that halts and
reports its partial work as an answer has converted a budget into a quality
problem.

## When to use this

Any agentic loop, particularly one that retries, and any agent that runs on a
schedule with nobody watching.

## Procedure

**1 — Enumerate every ceiling.** Steps, tokens, wall clock, cost, actions,
per-target attempts. For each, where it is checked and what it does on breach.

**2 — Order them by when they bind.** Run a task that consumes all resources
and record which fires first. A per-target ceiling usually binds long before a
token budget, which means the token budget was never the control.

**3 — Test the breach path.** The result must carry an explicit incomplete
flag. Check what a caller does with it: a loop that returns `complete: False`
into a pipeline that ignores the field has the same outcome as no budget.

**4 — Check the ceiling is not resettable by the agent.** A budget the loop can
extend on its own behalf — by starting a sub-task, spawning a child, or
retrying at a new target — is advisory.

**5 — Derive the numbers from observation.** State the p95 of a legitimate run
and set each ceiling above it. A round number either strangles real work or
never fires.

## Example

**Input** — the fixture committed at the top of [`scripts/budget_and_stop_condition_audit.py`](scripts/budget_and_stop_condition_audit.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
steps taken : 6
stopped by  : per_target (reports-db)
complete    : False   <- visible in the output, not silent

ceiling           limit     used
tokens            50000    10800
seconds              60      2.4
actions              20        6
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "ceilings": [{"name": "str", "value": 0, "checked_at": "str", "binds_at_step": 0}],
  "first_to_bind": "str",
  "breach": {"returns_incomplete": true, "caller_respects_flag": false},
  "agent_resettable": ["str"],
  "basis": {"p95_legitimate_run": {"steps": 0, "tokens": 0}}
}
```

## Failure modes

- **Listing budgets without ordering them.** Only the first one exists.
- **Returning partial work as an answer.** The flag is the control; the number
  is the trigger.
- **A ceiling the agent can reset by spawning.** Count the whole tree.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/runtime/budget-and-stop-condition-audit/scripts/budget_and_stop_condition_audit.py
SCRIPT = "skills/runtime/budget-and-stop-condition-audit/scripts/budget_and_stop_condition_audit.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The impossible task from A1.13 now stops after six steps, halted by the per-target ceiling — before the token or action budgets are anywhere near exhausted — and the result carries `complete: False` rather than reporting what it managed.

## Your turn

Check whether your agent's budget bounds calls per downstream target. If it only bounds tokens, your cost is protected and the service your agent hammers is not.

---

**Next → [A3.5 · Validating what comes back](https://spbreed.github.io/cyber-commons/lessons/A3.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*